# Week 5 Lab. Brownian Motion, Martingales and Trends
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v1 (2026-09-29)

---

### What this lab does

Today's deck asked three questions. This notebook answers each one with a simulation you run yourself. The seeds match the deck, so the numbers you print are the ones on the slides, and every slide marked **IN THE LAB** has a cell here that does what its badge says.

**Accumulate.** If random shocks keep adding up, how far can the series wander over time?
**Forecast.** If future shocks are unpredictable and average to zero, what should we predict for tomorrow?
**Persist.** If a shock hits today, does its effect disappear later, or permanently shift the level?

> 요약: 덱의 세 질문(충격이 쌓이면 얼마나 멀리 떠돌 수 있나 · 내일은 무엇을 예측해야 하나 · 충격의 효과가 남나 사라지나)을 직접 돌려 보는 실습입니다. 시드가 덱과 같아서 여기서 찍히는 숫자가 슬라이드와 같고, 「IN THE LAB」 표시가 있는 장마다 그 표시대로 해 보는 셀이 있습니다.

| Step | What you do |
|---|---|
| 1 | Shrink the steps of a random walk, draw what three step sizes do, and find the one that keeps the variance after one unit of time at 1 |
| 2 | Simulate Brownian paths, check $\mathrm{Var}\,W(t) = t$, and look at the $\pm\sqrt{t}$ band |
| 3 | Measure how strongly $W(2)$ and $W(5)$ move together |
| 4 | Forecast $W(5)$ from $W(2)$ and compare two forecasts |
| 5 | Build two rising series from the same shocks, then change one shock by +2 |
| 6 | Detrend and difference both series and read the lag-1 autocorrelation |
| 7 | Optional. Zooming in on a path, and a first look at next week |

### How to use it

Three cells are marked **`TODO`**. Those three are the part worth doing by hand.
직접 채워 보는 것이 이 노트북의 핵심입니다.

> Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; there is nothing to delete.
> If you run the cell before that, it stops with a `NotImplementedError` whose message says exactly this.
> **That is expected, not a broken notebook.** Each `TODO` is followed by a collapsed **Solution (정답)** cell:
> run it and the notebook continues from there. Open it only after you have tried.
> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다.

> ### Before you touch anything: **File > Save a copy in Drive**
> The link I posted opens **read-only**. Colab will say *"changes will not be saved"*.
> Save your own copy first, or everything you type here disappears when you close the tab.
> 먼저 내 드라이브에 사본을 저장하세요. 안 하면 입력한 내용이 탭을 닫을 때 사라집니다.
>
> **Nothing to submit for this notebook.** It is the voluntary lab block of Week 5. Today's topics are not in
> Assignment 1; they are in the midterm (Week 8).
> 이 노트북은 제출하지 않습니다. 오늘 내용은 과제 1 이 아니라 8주차 중간고사 범위입니다.

---
## 0. Setup

Nothing to install this week. Everything below ships with Colab.
설치할 것은 없습니다. 아래에 쓰는 것은 전부 Colab 에 이미 들어 있습니다.

**시드는 셀 안에서 정합니다.** 셀을 다시 실행해도 같은 숫자가 나오고, 그 숫자가 덱과 같습니다. 절 안에서는 첫 셀부터 차례로 실행하세요. 2~4절은 2절 첫 셀이 만든 경로 `W` 를 쓰므로, 런타임을 다시 시작했다면 그 셀부터 다시 실행하세요.
*Seeds are set inside the cells, so re-running a cell gives the same numbers, and they match the deck. Within a section, run the cells from the top. Sections 2 to 4 use the paths `W` made by the first cell of section 2, so after a runtime restart run that cell again first.*

그림의 축 이름은 영문으로 둡니다. 한글 축을 쓰려면 바로 아래 선택 셀을 먼저 실행하세요.
*Plot labels are kept in English. Run the optional cell below first if you want Korean labels.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

print("numpy:", np.__version__)

### Optional. Korean labels in plots / 그림에 한글 쓰기

Colab 의 기본 폰트에는 한글이 없어서 한글 축 이름이 네모로 깨집니다. 아래 셀을 한 번 실행하면 고쳐집니다.
*Colab's default font has no Korean glyphs. Run the cell below once if you want Korean axis labels.*

In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
# 실행하지 않아도 노트북은 전부 돌아갑니다. 축 이름이 영문이면 이 셀은 건너뛰세요.
# The notebook runs fine without this. Skip it if English labels are fine for you.
import subprocess, matplotlib, matplotlib.font_manager as fm

subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null", shell=True)
_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(_p)                           # 캐시를 다시 만들지 않고 바로 등록 / register without rebuilding the cache
plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
plt.rc("axes", unicode_minus=False)                  # 마이너스 기호 깨짐 방지 / keep the minus sign readable
print("한글 폰트 준비 완료 / Korean font ready")

---
## 1. 작은 걸음을 많이 / Many small steps

덱 「Many Small Steps Add Up」 과 같은 설정입니다. 시간 한 단위를 $n$ 조각으로 자르고, 조각 하나의 길이를 $h = 1/n$ 이라 합니다.
조각마다 동전을 던져 $+a$ 또는 $-a$ 만큼 움직입니다. 한 단위 시간 뒤의 위치는 걸음 $n$ 개의 합 $S = X_1 + \dots + X_n$ 입니다.
*Same setup as the deck slide "Many Small Steps Add Up". Cut one unit of time into $n$ pieces of length $h = 1/n$.
At each piece a fair coin moves the walk up by $a$ or down by $a$. The position after one unit of time is the sum of the $n$ steps.*

걸음이 독립이므로 분산이 더해집니다. $\mathrm{Var}(X_i) = a^2$ 이고, 걸음이 $1/h$ 개이므로
*The steps are independent, so the variances add. Each step has variance $a^2$ and there are $1/h$ of them, so*

$$\mathrm{Var}(S) = \frac{a^2}{h}.$$

**먼저 예측하세요.** $a = h$ 로 두면 $h$ 를 줄일 때 $\mathrm{Var}(S)$ 는 어떻게 되나요? $a = 1$ 이면요?
*Predict first. With $a = h$, what happens to $\mathrm{Var}(S)$ as $h$ shrinks? With $a = 1$?*

In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다 / Names used below
#   h        : 걸음 하나가 걸리는 시간. 한 단위 시간을 n 조각으로 자른 길이, h = 1/n
#              time taken by one step. One unit of time cut into n pieces, h = 1/n
#   n_paths  : 경로 수. 같은 규칙으로 여러 번 뽑는다 / number of paths, all drawn with the same rule
#   rng      : 난수 생성기 / random number generator
#   n        : 한 단위 시간 안의 걸음 수 / number of steps in one unit of time
#   signs    : 걸음마다 +1(위) 또는 -1(아래). 경로 x 걸음 표
#              +1 (up) or -1 (down) for every step. A table of paths x steps
#   a        : 걸음 크기. 한 걸음에 +a 또는 -a 만큼 움직인다   <- 여기가 빈칸
#              step size. Each step moves by +a or -a   <- this is the blank
#   S        : 한 단위 시간 뒤의 위치, 경로마다 하나 / position after one unit of time, one per path
#
# 참고 / Reference   rng.choice([-1.0, 1.0], size=(r, c))
#   -1 과 +1 중 하나를 반반 확률로 골라 r 행 c 열 표를 채운다.
#   Fills an r-by-c table with -1 or +1, each with probability 1/2.
#   예 / e.g.   rng.choice([-1.0, 1.0], size=(2, 3))  ->  [[ 1. -1.  1.] [-1. -1.  1.]] 같은 표 / a table like this
#
# 참고 / Reference   np.sqrt(x)
#   x 의 제곱근 / the square root of x.   예 / e.g.   np.sqrt(9.0)  ->  3.0

def position_after_one_unit(h, n_paths, rng):
    """Return S, the position after one unit of time, for n_paths random walks whose steps take time h."""
    n = int(round(1 / h))                                   # 한 단위 시간 안의 걸음 수 / steps in one unit of time
    signs = rng.choice([-1.0, 1.0], size=(n_paths, n))      # 걸음마다 위(+1) 또는 아래(-1) / up (+1) or down (-1)

    # TODO 1. Var(S) = a^2 / h 가 h 와 상관없이 1 이 되도록 걸음 크기 a 를 정하세요. 한 줄입니다.
    #         덱 「Why We Choose Steps of Size √h」의 결론입니다.
    # TODO 1. Choose the step size a so that Var(S) = a^2 / h equals 1 whatever h is. One line.
    #         This is the conclusion of the deck slide "Why We Choose Steps of Size √h".
    a = None
    if a is None:
        raise NotImplementedError(BLANK_MSG)

    S = (a * signs).sum(axis=1)                             # 걸음을 모두 더한 것이 위치 / the position is the sum of the steps
    return S
# --------------------------------------------------------------------------


rng = np.random.default_rng(5)                              # 이 절의 시드 / this section's seed
for h in (1 / 4, 1 / 16, 1 / 64, 1 / 256):
    S = position_after_one_unit(h, n_paths=4000, rng=rng)
    print("h = 1/%-4d  S 의 표본분산 / sample variance of S : %.4f" % (round(1 / h), S.var(ddof=1)))
# 기대값 / Expected
#   h = 1/4     S 의 표본분산 / sample variance of S : 1.0064
#   h = 1/16    S 의 표본분산 / sample variance of S : 0.9899
#   h = 1/64    S 의 표본분산 / sample variance of S : 1.0224
#   h = 1/256   S 의 표본분산 / sample variance of S : 1.0268
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다. 넷 다 1 근처이면 맞습니다.
#   The seed is fixed, so you should see exactly these numbers. All four near 1 means it is right.
#   0.25, 0.0625, 0.0156, 0.0039 처럼 h 와 같은 값이면 a = h 를 넣은 것입니다. 경로가 납작해집니다.
#   Values equal to h (0.25, 0.0625, ...) mean a = h went in. The path flattens out.
#   4, 16, 64, 256 근처면 a = 1 을 넣은 것입니다. 퍼짐이 끝없이 커집니다.
#   Values near 4, 16, 64, 256 mean a = 1 went in. The spread grows without bound.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def position_after_one_unit(h, n_paths, rng):
    """Return S, the position after one unit of time, for n_paths random walks whose steps take time h."""
    n = int(round(1 / h))                                   # 한 단위 시간 안의 걸음 수 / steps in one unit of time
    signs = rng.choice([-1.0, 1.0], size=(n_paths, n))      # 걸음마다 위(+1) 또는 아래(-1) / up (+1) or down (-1)

    a = np.sqrt(h)               # 빈칸이었던 줄. a^2 / h = h / h = 1 이 되는 유일한 선택
                                 # the blank. The only choice that makes a^2 / h = h / h = 1
    if a is None:
        raise NotImplementedError(BLANK_MSG)

    S = (a * signs).sum(axis=1)                             # 걸음을 모두 더한 것이 위치 / the position is the sum of the steps
    return S



rng = np.random.default_rng(5)                              # 이 절의 시드 / this section's seed
for h in (1 / 4, 1 / 16, 1 / 64, 1 / 256):
    S = position_after_one_unit(h, n_paths=4000, rng=rng)
    print("h = 1/%-4d  S 의 표본분산 / sample variance of S : %.4f" % (round(1 / h), S.var(ddof=1)))
# 기대값 / Expected
#   h = 1/4     S 의 표본분산 / sample variance of S : 1.0064
#   h = 1/16    S 의 표본분산 / sample variance of S : 0.9899
#   h = 1/64    S 의 표본분산 / sample variance of S : 1.0224
#   h = 1/256   S 의 표본분산 / sample variance of S : 1.0268
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다. 넷 다 1 근처이면 맞습니다.
#   The seed is fixed, so you should see exactly these numbers. All four near 1 means it is right.
#   0.25, 0.0625, 0.0156, 0.0039 처럼 h 와 같은 값이면 a = h 를 넣은 것입니다. 경로가 납작해집니다.
#   Values equal to h (0.25, 0.0625, ...) mean a = h went in. The path flattens out.
#   4, 16, 64, 256 근처면 a = 1 을 넣은 것입니다. 퍼짐이 끝없이 커집니다.
#   Values near 4, 16, 64, 256 mean a = 1 went in. The spread grows without bound.

In [ ]:
# 자가검증 / Self-check
# 표본분산은 표본마다 흔들립니다. 경로 4000 개면 그 흔들림의 규모는 sqrt(2 / 3999), 약 0.022 입니다.
# A sample variance wobbles from sample to sample. With 4000 paths its typical size is sqrt(2 / 3999), about 0.022.
se = np.sqrt(2 / 3999)
for h in (1 / 4, 1 / 64):
    v = position_after_one_unit(h, 4000, np.random.default_rng(11)).var(ddof=1)
    assert abs(v - h) > 5 * se, "분산이 h 와 같습니다. a = h 를 넣지 않았나요? / The variance equals h. Did a = h go in?"
    assert abs(v - 1 / h) > 5 * se, "분산이 1/h 입니다. a = 1 을 넣지 않았나요? / The variance is 1/h. Did a = 1 go in?"
    assert abs(v - 1) < 5 * se, ("h = %g 에서 분산 %.3f, 1 에서 너무 멉니다 / at h = %g the variance %.3f is too far from 1"
                                 % (h, v, h, v))
print("통과. 걸음을 sqrt(h) 로 줄이면 한 단위 시간 뒤의 퍼짐이 h 와 상관없이 1 입니다.")
print("Passed. Shrink the step like sqrt(h) and the spread after one unit of time is 1 whatever h is.")

**What to notice.** A step proportional to $\sqrt{h}$ keeps a spread that is neither 0 nor infinite ($2\sqrt{h}$ would leave variance 4).
Choosing exactly $a = \sqrt{h}$ keeps the variance after one unit of time at 1. That is the standard scaling, and it is why rule 3 of
the deck says a change over time $h$ has variance $h$.

> 요약: 걸음이 $\sqrt{h}$ 에 비례해야 퍼짐이 0 도 무한대도 아니게 남고, 정확히 $\sqrt{h}$ 면 한 단위 시간 뒤 분산이 1 입니다. 브라운 운동의 규칙 3 (구간 길이 = 분산) 이 여기서 옵니다.

**Try it. 세 걸음 크기를 모두 그려 보기.** 덱 「Why We Choose Steps of Size √h」 의 표를 그림으로 봅니다. 같은 동전 던지기에 걸음 크기만 $a = h$, $a = \sqrt{h}$, $a = 1$ 로 바꿔 경로를 그립니다. 가로는 $h$ 를 1/16, 1/64, 1/256 으로 줄인 것이고, 한 줄 안에서는 세로축을 같게 두었습니다.
*The table on "Why We Choose Steps of Size √h", drawn as paths. The same coin tosses, with only the step size changed to $a = h$, $a = \sqrt{h}$ and $a = 1$. Across a row $h$ shrinks through 1/16, 1/64, 1/256, and the three panels of a row share one vertical axis.*

**먼저 예측하세요.** $h$ 를 줄이면 윗줄($a = h$)의 경로는 어떻게 보일까요? 아랫줄($a = 1$)은요? 가운데 줄은요?
*Predict first. As $h$ shrinks, what happens to the paths in the top row ($a = h$)? In the bottom row ($a = 1$)? In the middle row?*

In [ ]:
# 세 걸음 크기 비교. 같은 h 에서는 세 줄이 같은 동전 던지기(signs)를 쓰고 걸음 크기 a 만 다르다.
# Three step sizes. For each h the three rows use the same coin tosses (signs); only the step size a differs.
three_scalings = {"a = h": lambda h: h, "a = sqrt(h)": lambda h: np.sqrt(h), "a = 1": lambda h: 1.0}
hs = (1 / 16, 1 / 64, 1 / 256)

fig, ax = plt.subplots(3, 3, figsize=(12, 7.5), sharex=True, sharey="row")
for j, h in enumerate(hs):
    n = int(round(1 / h))                                   # 한 단위 시간 안의 걸음 수 / steps in one unit of time
    signs = np.random.default_rng(8).choice([-1.0, 1.0], size=(20, n))   # 경로 20 개 / 20 paths
    grid = np.linspace(0, 1, n + 1)                         # 시각 0, h, 2h, …, 1 / the time grid
    for r, (name, step) in enumerate(three_scalings.items()):
        a = step(h)
        paths = np.c_[np.zeros(20), np.cumsum(a * signs, axis=1)]   # 0 에서 출발해 걸음을 쌓는다 / start at 0, add up the steps
        ax[r, j].plot(grid, paths.T, lw=0.7, alpha=0.7)
        ax[r, j].set_title("%s,  h = 1/%d,  Var(S) = %s" % (name, n, ("%.4f" % (a * a / h)).rstrip("0").rstrip(".")))
    ax[2, j].set_xlabel("time")
for r, name in enumerate(three_scalings):
    ax[r, 0].set_ylabel(name)
plt.tight_layout()
plt.show()

# 그림 아래 숫자. 경로 4000 개로 S 의 표본분산을 재서 이론값 a^2 / h 와 나란히 찍는다.
# The numbers under the picture. The sample variance of S over 4000 paths, next to the theory value a^2 / h.
print((" " * 12 + "".join(("h = 1/%d" % round(1 / h)).ljust(22) for h in hs)).rstrip())
for name, step in three_scalings.items():
    row = ""
    for h in hs:
        n = int(round(1 / h))
        S = (step(h) * np.random.default_rng(9).choice([-1.0, 1.0], size=(4000, n))).sum(axis=1)
        theory = ("%.4f" % (step(h) ** 2 / h)).rstrip("0").rstrip(".")
        row += ("%.4f  (%s)" % (S.var(ddof=1), theory)).ljust(22)
    print(name.ljust(12) + row.rstrip())
# 기대값 / Expected   표본분산 (이론값) / sample variance (theory)
#               h = 1/16              h = 1/64              h = 1/256
#   a = h       0.0633  (0.0625)      0.0154  (0.0156)      0.0039  (0.0039)
#   a = sqrt(h) 1.0127  (1)           0.9827  (1)           1.0007  (1)
#   a = 1       16.2026  (16)         62.8900  (64)         256.1699  (256)
#   윗줄 a = h      : 분산이 h 와 같이 0 으로 간다. h 를 줄일수록 경로가 0 에 눌려 납작해진다.
#   Top, a = h      : the variance is h and goes to 0. The smaller h, the flatter the paths around 0.
#   가운데 a = √h   : 분산이 늘 1. h 를 줄여도 세로 폭이 그대로이고 경로만 더 잘게 흔들린다.
#   Middle, a = √h  : the variance stays 1. The vertical scale holds while the path gets finer.
#   아랫줄 a = 1    : 분산이 1/h = n. h 를 1/4 로 줄일 때마다 세로 폭이 두 배(√n)로 커진다.
#   Bottom, a = 1   : the variance is 1/h = n. Each time h shrinks by 4 the vertical scale doubles (√n).


**What to notice.** 세 줄이 쓰는 동전 던지기는 같습니다. 다른 것은 걸음 크기 하나뿐인데, $h$ 를 줄이면 $a = h$ 는 0 으로 납작해지고, $a = 1$ 은 폭이 끝없이 커지고, 셋 가운데 $a = \sqrt{h}$ 만 폭이 그대로 남습니다. 표의 $\mathrm{Var}(S) \to 0$, $= 1$, $\to \infty$ 가 이 세 모양입니다.
*The three rows use the same coin tosses. Only the step size differs, and as $h$ shrinks $a = h$ flattens to 0, $a = 1$ blows up, and of the three only $a = \sqrt{h}$ keeps its vertical scale. The table's $\mathrm{Var}(S) \to 0$, $= 1$ and $\to \infty$ are these three pictures.*

> 요약: 극한에서 0 도 무한대도 아닌 경로가 남는 것은 걸음이 $\sqrt{h}$ 에 비례할 때뿐이고($2\sqrt{h}$ 면 분산 4), 정확히 $\sqrt{h}$ 면 한 단위 시간 뒤 분산이 1 입니다. 표준 브라운 운동이 이 스케일로 정의되는 이유입니다.

**Try it.** 덱 「Why Do Many Small Steps Make a Bell Curve?」 의 그림입니다. 걸음 수 $n$ 을 4, 16, 64 으로 늘리면서 한 단위 시간 뒤의 위치 $S$ 의 히스토그램을 $N(0, 1)$ 곡선과 겹쳐 보세요. 막대 면적이 확률이고, 분산은 어느 $n$ 에서도 1 입니다. 모양만 정규분포로 다가갑니다.
*The figure on "Why Do Many Small Steps Make a Bell Curve?". Increase the number of steps $n$ through 4, 16, 64 and overlay the histogram of $S$ on the $N(0, 1)$ curve. Bar areas are probabilities, the variance is 1 for every $n$, and only the shape moves toward the normal.*

In [ ]:
# 덱 그림과 같은 실험. 위에서 만든 position_after_one_unit 을 그대로 쓴다 (a = sqrt(h) 이므로 Var(S) = 1).
# Same experiment as the deck figure, reusing position_after_one_unit from above (a = sqrt(h), so Var(S) = 1).
xs = np.linspace(-3.5, 3.5, 300)
pdf = np.exp(-xs ** 2 / 2) / np.sqrt(2 * np.pi)              # N(0, 1) 밀도 / the N(0, 1) density
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for a_, n in zip(ax, (4, 16, 64)):
    S = position_after_one_unit(1 / n, 20_000, np.random.default_rng(n))
    width = 2 / np.sqrt(n)                                  # S 가 밟는 값의 간격 2a. 막대 하나가 값 하나 / spacing 2a of the values S can take, one bar per value
    m = np.arange(-4 / width, 4 / width + 1)
    a_.hist(S, bins=(m - 0.5) * width, density=True, alpha=0.6)
    a_.plot(xs, pdf, "k-", lw=1)
    a_.set_title("n = %d steps, Var(S) = %.2f" % (n, S.var(ddof=1)))
plt.tight_layout()
plt.show()
# 기대값 / Expected
#   n = 4 는 값이 다섯 개뿐이라 막대 다섯 개, n = 64 는 곡선에 거의 붙습니다. 분산은 셋 다 1 근처입니다.
#   n = 4 has only five possible values, so five bars; n = 64 sits almost on the curve. All three variances are near 1.


---
## 2. 브라운 경로와 퍼짐 / Brownian paths and their spread

이제 동전 대신 정규분포 걸음을 씁니다. 덱 「Definition of Standard Brownian Motion」 의 규칙 3 에 따르면 길이 $dt$ 인 구간의 변화는 $N(0, dt)$ 이므로,
걸음 하나는 표준정규 난수에 $\sqrt{dt}$ 를 곱한 것입니다. 1절의 $\sqrt{h}$ 와 같은 스케일입니다.
*Now the steps are normal instead of coin flips. Rule 3 on the deck slide "Definition of Standard Brownian Motion" says a change over
time $dt$ is $N(0, dt)$, so one step is a standard normal number times $\sqrt{dt}$. Same scaling as $\sqrt{h}$ in section 1.*

이 셀의 코드는 덱 Lab 1 슬라이드와 같고, 시드도 덱 「One Process, Many Possible Paths」 그림과 같습니다.
*The code below is the same as on the deck's Lab 1 slide, and the seed is the one behind the figure on "One Process, Many Possible Paths".*

**먼저 예측하세요.** $\mathrm{Var}\,W(1)$ 과 $\mathrm{Var}\,W(4)$ 는 얼마여야 하나요? $\mathrm{Var}\,W(t)$ 를 $t$ 에 대해 그리면 기울기는?
*Predict first. What should $\mathrm{Var}\,W(1)$ and $\mathrm{Var}\,W(4)$ be? What slope should $\mathrm{Var}\,W(t)$ have against $t$?*

In [ ]:
rng = np.random.default_rng(20260929)                      # 수업 날짜가 시드. 덱 그림과 같은 경로 / the class date, same paths as the deck
n_paths, n_steps, T = 2000, 250, 5.0                        # 경로 2000 개, 0 부터 5 까지 250 걸음 / 2000 paths, 250 steps from 0 to 5
dt = T / n_steps                                            # 걸음 하나의 시간 0.02 / time per step, 0.02
dW = np.sqrt(dt) * rng.standard_normal((n_paths, n_steps))  # 걸음 = sqrt(dt) x 표준정규 / step = sqrt(dt) x standard normal
W = np.c_[np.zeros(n_paths), np.cumsum(dW, axis=1)]         # W(0) = 0 을 앞에 붙이고 걸음을 쌓는다 / start at 0, add up the steps
t = np.linspace(0, T, n_steps + 1)                          # 시각 0, 0.02, …, 5 / the time grid
var_t = W.var(axis=0, ddof=1)                               # 시각마다 경로 2000 개의 분산 / one variance per time, over the 2000 paths
print(np.polyfit(t, var_t, 1)[0])                           # 분산을 시각에 대해 직선으로 맞춘 기울기 / the slope of variance against time

print("Var W(1) =", round(var_t[50], 3), "   Var W(4) =", round(var_t[200], 3))
# 기대값 / Expected
#   0.9678514553909199
#   Var W(1) = 1.084    Var W(4) = 3.917
#   기울기가 1 근처, Var W(1) 이 1 근처, Var W(4) 가 4 근처면 맞습니다. 경로 2000 개라 몇 퍼센트는 흔들립니다.
#   A slope near 1, Var W(1) near 1 and Var W(4) near 4 is right. With 2000 paths a few percent of wobble is normal.

In [ ]:
# 덱 「Variance and Standard Deviation Grow Differently」 의 그림. 분산은 t 로, 표준편차는 sqrt(t) 로 큽니다.
# The figure on "Variance and Standard Deviation Grow Differently". The variance grows like t, the SD only like sqrt(t).
#   검은 선과 점 : 이론. 덱처럼 t = 16 까지, 표의 네 시각 1, 4, 9, 16 에 점
#   Black        : theory up to t = 16 as on the deck, with dots at the table's times 1, 4, 9, 16
#   색 선        : 2절 첫 셀의 경로 2000 개로 잰 값. 시뮬레이션이 t = 5 에서 끝나 거기까지
#   Coloured     : measured over the 2000 paths of section 2, up to t = 5 where the simulation ends
tg = np.linspace(0, 5.0, W.shape[1])                        # 2절의 시각 0, 0.02, …, 5. 다른 절이 t 를 다시 쓰므로 여기서 따로 만든다
                                                            # section 2's time grid, rebuilt here because other sections reuse the name t
th = np.linspace(0, 16, 321)                                # 이론 곡선의 시각 0 … 16 / times for the theory curves
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(th, th, "k-", lw=1, label="Var W(t) = t")
ax.plot(th, np.sqrt(th), "k--", lw=1, label="SD = sqrt(t)")
ax.plot([1, 4, 9, 16], [1, 4, 9, 16], "ko", ms=4)           # 표의 분산 1, 4, 9, 16 / the table's variances
ax.plot([1, 4, 9, 16], [1, 2, 3, 4], "ko", ms=4)            # 표의 표준편차 1, 2, 3, 4 / the table's SDs
ax.plot(tg, var_t, lw=2, label="sample Var W(t), 2000 paths")
ax.plot(tg, W.std(axis=0, ddof=1), lw=2, label="sample SD of W(t), 2000 paths")   # 시각마다 표준편차 하나 / one SD per time
ax.set_xticks([0, 1, 4, 9, 16])
ax.set_xlabel("time t")
ax.set_title("variance grows like t, SD only like sqrt(t)")
ax.legend()
plt.show()
# 기대값 / Expected
#   색 선이 t = 5 까지 검은 선 위에 놓입니다. t = 4 에서 분산은 4, 표준편차는 2 근처입니다.
#   The coloured lines sit on the black ones up to t = 5. At t = 4 the variance is near 4 and the SD near 2.
#   시간이 1 에서 4 로 네 배가 되면 분산은 네 배, 표준편차는 두 배입니다. t = 9 와 16 은 이론 곡선에서 읽습니다.
#   Going from t = 1 to t = 4 gives four times the variance but only twice the SD. Read t = 9 and 16 off the theory curves.

In [ ]:
# 덱 「One Process, Many Possible Paths」 의 그림. 0~29 번 경로(9 번을 진하게)와, t = 2 에서 잰 경로 2000 개의 값 W(2).
# The figure on "One Process, Many Possible Paths". Paths 0 to 29 (path 9 dark) and the 2000 values of W(2) at t = 2.
tg = np.linspace(0, 5.0, W.shape[1])                        # 2절의 시각 격자 / section 2's time grid
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True, gridspec_kw={"width_ratios": [3, 1]})
for k in range(30):
    ax[0].plot(tg, W[k], color="0.75", lw=0.6)              # 경로 30 개 / thirty paths
ax[0].plot(tg, W[9], color="darkgreen", lw=1.4)             # 덱에서 진하게 그린 경로 하나 / the one path the deck draws dark
ax[0].axvline(2, color="goldenrod", ls="--", lw=1)          # t = 2
ax[0].set_title("30 paths, one of them dark")
ax[0].set_xlabel("time t")
ax[1].hist(W[:, 100], bins=30, orientation="horizontal", color="0.6")   # t = 2 는 100 번 칸 / index 100 is t = 2
ax[1].set_title("2,000 values of W(2)")
plt.tight_layout()
plt.show()
# 기대값 / Expected
#   오른쪽 막대가 W(2) 의 분포입니다. 0 근처가 가장 높고, 대부분 -3 과 3 사이에 있습니다(표준편차 sqrt(2), 약 1.41).
#   The bars on the right are the distribution of W(2). They are tallest near 0 and mostly lie between -3 and 3 (SD sqrt(2), about 1.41).

### 띠는 한 시각의 퍼짐입니다 / The band is the spread at one time

정규분포에서 표준편차 하나 안에는 약 68% 가 들어갑니다. 그러면 경로의 68% 가 $\pm\sqrt{t}$ 띠 안에 **계속** 머물까요?
*For a normal distribution about 68% lies within one SD. Does that mean 68% of the paths stay inside the $\pm\sqrt{t}$ band the whole time?*

In [ ]:
tg = np.linspace(0, 5.0, W.shape[1])                        # 2절의 시각 격자. 위 그림 셀과 같다 / section 2's time grid, as in the plot cell above
band = np.sqrt(tg)                                          # 시각마다 표준편차 하나 / one SD at each time
inside_at_4 = np.abs(W[:, 200]) <= band[200]                # t = 4 에서 띠 안인가 / inside the band at t = 4?
left_once = np.any(np.abs(W[:, 50:]) > band[50:], axis=1)   # 1 <= t <= 5 에서 한 번이라도 나갔나 / ever outside on 1 <= t <= 5?

print("t = 4 에서 띠 안   / inside at t = 4          :", inside_at_4.sum(), "of", n_paths)
print("한 번이라도 나감   / left at least once, 1-5  :", left_once.sum(), "of", n_paths)
# 기대값 / Expected
#   t = 4 에서 띠 안   / inside at t = 4          : 1375 of 2000
#   한 번이라도 나감   / left at least once, 1-5  : 1707 of 2000
#   덱 「A One-SD Band Is Not a Boundary」 의 두 숫자와 같습니다. 같은 경로이기 때문입니다.
#   The same two numbers as on "A One-SD Band Is Not a Boundary", because these are the same paths.
#   둘째 숫자는 0.02 간격 격자에서 센 것입니다. 격자를 촘촘히 하면 더 커집니다.
#   The second number is counted on a grid of step 0.02. A finer grid makes it larger.

# 덱 「A One-SD Band Is Not a Boundary」 의 그림. 덱은 20~39 번 경로를 그리고, 띠를 한 번 이상 나간 21 번을 빨갛게 칠했습니다.
# The figure on "A One-SD Band Is Not a Boundary". The deck draws paths 20 to 39 and colors path 21, one that leaves the band, red.
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.fill_between(tg, -band, band, color="0.88", label="+-sqrt(t) band")   # 한 시각의 표준편차 하나 / one SD at each time
for k in range(20, 40):
    ax.plot(tg, W[k], color="0.6", lw=0.6)                  # 경로 20 개 / twenty paths
ax.plot(tg, W[21], color="firebrick", lw=1.4, label="path 21, leaves the band")
ax.plot(tg, 2 * band, "k--", lw=1, label="+-2 sqrt(t)")
ax.plot(tg, -2 * band, "k--", lw=1)
ax.axvline(4, color="goldenrod", ls="--", lw=1)             # t = 4
ax.set_title("the band is not a wall")
ax.set_xlabel("time t")
ax.legend(loc="upper left")
plt.show()

**What to notice.** At one fixed time about 68% of the paths are inside $\pm\sqrt{t}$, as the normal law says.
Over a stretch of time most paths leave the band at least once. A probability at one time is not a promise about the whole path.

> 요약: 한 시각에서는 약 68% 가 띠 안에 있지만, 1 에서 5 까지 한 번이라도 나가는 경로가 대부분입니다. 띠는 벽이 아닙니다.

**Try it.** 위 첫 셀에서 `np.sqrt(dt)` 를 `dt` 로 바꾸고 다시 실행해 보세요. 기울기가 얼마로 바뀌나요? 1절의 $a = h$ 와 같은 이야기입니다.
*Replace `np.sqrt(dt)` by `dt` in the first cell of this section and run it again. What does the slope become? It is the $a = h$ case of section 1.*

해 본 뒤에는 `dt` 를 `np.sqrt(dt)` 로 되돌리고 그 셀을 다시 실행한 다음 넘어가세요. 3·4절의 숫자와 설명은 표준 브라운 운동(걸음이 $\sqrt{dt}$ 배)을 전제로 합니다.
*Afterwards change `dt` back to `np.sqrt(dt)` and re-run that cell before you move on. The numbers and explanations in sections 3 and 4 assume standard Brownian motion (steps scaled by $\sqrt{dt}$).*

---
## 3. 공유한 역사 / Shared history

덱 「Compute a Brownian Covariance」 에서 네 줄로 $\mathrm{Cov}(W(2), W(5)) = \mathrm{Var}\,W(2) = 2 = \min(2, 5)$ 를 얻었습니다.
$W(5)$ 가 $W(2)$ 에 겹치지 않는 새 변화 $W(5) - W(2)$ 를 더한 것이고, 그 새 변화는 $W(2)$ 와 독립이기 때문입니다.
*On "Compute a Brownian Covariance" four lines gave $\mathrm{Cov}(W(2), W(5)) = 2 = \min(2, 5)$: $W(5)$ is $W(2)$ plus a new change that is independent of $W(2)$.*

**먼저 예측하세요.** 경로 2000 개에서 잰 공분산과 상관계수는 얼마쯤일까요?
*Predict first. Roughly what covariance and correlation will 2000 paths give?*

In [ ]:
W2, W5 = W[:, 100], W[:, 250]                               # 격자 간격 0.02 라서 t = 2 는 100 번, t = 5 는 250 번 / grid step 0.02
cov_hat = np.mean((W2 - W2.mean()) * (W5 - W5.mean()))     # 2주차의 정의 그대로. 편차 곱의 평균 / Week 2 definition: mean product of deviations
corr_hat = cov_hat / (W2.std() * W5.std())                  # 공분산을 두 표준편차로 나눈다 / covariance over the two SDs

print("공분산   / covariance  :", round(cov_hat, 3), "   이론 / theory  min(2, 5) = 2")
print("상관계수 / correlation :", round(corr_hat, 3), "   이론 / theory  2 / sqrt(10) =", round(2 / np.sqrt(10), 3))
# 기대값 / Expected
#   공분산   / covariance  : 2.011    이론 / theory  min(2, 5) = 2
#   상관계수 / correlation : 0.64    이론 / theory  2 / sqrt(10) = 0.632
#   2 와 0.632 근처면 맞습니다. 차이는 경로 2000 개의 표본오차입니다.
#   Near 2 and 0.632 is right. The gap is sampling error from 2000 paths.

**What to notice.** The only part the two values share is the history on $[0, 2]$, and its variance is 2. The new changes on $(2, 5]$ add spread to $W(5)$ but no covariance.

> 요약: 공분산은 공유한 역사의 길이, $\min(s, t)$ 입니다. 새 변화는 퍼짐만 더하고 공분산은 더하지 않습니다.

---
## 4. 지금 아는 것으로 예측하기 / Forecast from what you know

$t = 2$ 에 서서 $W(2)$ 를 보았습니다. $W(5)$ 의 평균 예측으로 무엇을 내놓아야 할까요?
덱 「Why Brownian Motion Is a Martingale」 은 $E[W(t) \mid W(u),\ u \le s] = W(s)$ 를 네 줄로 보였습니다.
*Standing at $t = 2$ you have seen $W(2)$. What should the mean forecast of $W(5)$ be?
The deck slide "Why Brownian Motion Is a Martingale" showed in four lines that the conditional mean of a future value is the current value.*

두 예측을 비교합니다. 하나는 여러분이 채울 예측, 다른 하나는 과거를 무시하고 전체 평균 0 을 내는 예측입니다.
*We compare two forecasts: the one you fill in, and one that ignores the past and always says 0, the overall mean.*

비교 기준은 평균제곱오차(MSE)입니다. 틀린 크기를 제곱해서 벌점을 줄 때, 분산이 유한하면 조건부 평균이 기대 벌점을 가장 작게 만듭니다. 「최선의 평균 예측」은 이 뜻입니다.
*The yardstick is the mean squared error. With finite second moments, the conditional mean gives the smallest expected squared error; that is what "best mean forecast" means.*

이 절은 2절 첫 셀이 만든 경로 `W` 를 씁니다. 런타임을 다시 시작했다면 그 셀부터 실행하세요.
*This section uses the paths `W` made by the first cell of section 2. After a runtime restart, run that cell first.*

In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다 / Names used below
#   W2       : 경로마다 t = 2 에서의 값. 지금까지 본 것 / the value at t = 2 on each path, what we have seen
#   W5       : 경로마다 t = 5 에서의 값. 맞히려는 미래 / the value at t = 5 on each path, the future we forecast
#   forecast : t = 2 에 서서 내놓는 W(5) 의 평균 예측, 경로마다 하나   <- 여기가 빈칸
#              the mean forecast of W(5) made at t = 2, one per path   <- this is the blank
#   mse      : 예측 오차 제곱의 평균. 작을수록 좋은 평균 예측 / mean squared error, smaller is better

def mean_forecast_of_W5(W2):
    """Given the values at t = 2, return the mean forecast of W(5), one per path."""
    # TODO 2. 브라운 운동은 마팅게일입니다. 지금까지 본 값이 W2 일 때 W(5) 의 조건부 평균은 무엇인가요?
    #         그 값을 forecast 에 넣으세요. 한 줄입니다.
    # TODO 2. Brownian motion is a martingale. Having seen W2, what is the conditional mean of W(5)?
    #         Put it into forecast. One line.
    forecast = None
    if forecast is None:
        raise NotImplementedError(BLANK_MSG)

    return forecast
# --------------------------------------------------------------------------


W2, W5 = W[:, 100], W[:, 250]                               # 3절과 같은 두 시각. 이 절만 따로 돌려도 되게
                                                            # the same two times as section 3, so this section runs on its own
fc = mean_forecast_of_W5(W2)
mse_fc = np.mean((W5 - fc) ** 2)                            # 채운 예측의 평균제곱오차 / mean squared error of your forecast
mse_zero = np.mean((W5 - 0.0) ** 2)                         # 늘 0 이라고 하는 예측 / the forecast that always says 0
print("여러분의 예측 / your forecast     MSE :", round(mse_fc, 3), "   이론 / theory  Var(W(5) - W(2)) = 3")
print("늘 0 인 예측  / always 0          MSE :", round(mse_zero, 3), "   이론 / theory  Var W(5) = 5")
# 기대값 / Expected
#   여러분의 예측 / your forecast     MSE : 2.827    이론 / theory  Var(W(5) - W(2)) = 3
#   늘 0 인 예측  / always 0          MSE : 4.788    이론 / theory  Var W(5) = 5
#   첫 줄이 3 근처, 둘째 줄이 5 근처면 맞습니다. 과거를 쓰면 오차가 5 에서 3 으로 줄어듭니다.
#   Near 3 on the first line and near 5 on the second is right. Using the past cuts the error from 5 to 3.
#   첫 줄도 5 근처면 W2 의 평균(숫자 하나)처럼 과거를 버리는 값을 넣은 것입니다.
#   A first line near 5 as well means something that throws the past away, such as the mean of W2, went in.
#   3 보다 크면 2.5 * W2 처럼 시간 비율로 늘린 것입니다. 미래 변화의 평균은 0 이라 늘리면 안 됩니다.
#   Above 3 means something like 2.5 * W2 went in. The future change has mean 0, so do not stretch it.
#   첫 줄이 0 이면 아직 모르는 W5 를 넣은 것입니다. t = 2 에서는 W(5) 를 모릅니다.
#   A first line of 0 means W5 went in. At t = 2 we do not know W(5) yet.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def mean_forecast_of_W5(W2):
    """Given the values at t = 2, return the mean forecast of W(5), one per path."""
    forecast = W2                # 빈칸이었던 줄. 마팅게일이라 미래의 조건부 평균이 곧 지금 값이다
                                 # the blank. For a martingale the conditional mean of the future is the current value
    if forecast is None:
        raise NotImplementedError(BLANK_MSG)

    return forecast



W2, W5 = W[:, 100], W[:, 250]                               # 3절과 같은 두 시각. 이 절만 따로 돌려도 되게
                                                            # the same two times as section 3, so this section runs on its own
fc = mean_forecast_of_W5(W2)
mse_fc = np.mean((W5 - fc) ** 2)                            # 채운 예측의 평균제곱오차 / mean squared error of your forecast
mse_zero = np.mean((W5 - 0.0) ** 2)                         # 늘 0 이라고 하는 예측 / the forecast that always says 0
print("여러분의 예측 / your forecast     MSE :", round(mse_fc, 3), "   이론 / theory  Var(W(5) - W(2)) = 3")
print("늘 0 인 예측  / always 0          MSE :", round(mse_zero, 3), "   이론 / theory  Var W(5) = 5")
# 기대값 / Expected
#   여러분의 예측 / your forecast     MSE : 2.827    이론 / theory  Var(W(5) - W(2)) = 3
#   늘 0 인 예측  / always 0          MSE : 4.788    이론 / theory  Var W(5) = 5
#   첫 줄이 3 근처, 둘째 줄이 5 근처면 맞습니다. 과거를 쓰면 오차가 5 에서 3 으로 줄어듭니다.
#   Near 3 on the first line and near 5 on the second is right. Using the past cuts the error from 5 to 3.
#   첫 줄도 5 근처면 W2 의 평균(숫자 하나)처럼 과거를 버리는 값을 넣은 것입니다.
#   A first line near 5 as well means something that throws the past away, such as the mean of W2, went in.
#   3 보다 크면 2.5 * W2 처럼 시간 비율로 늘린 것입니다. 미래 변화의 평균은 0 이라 늘리면 안 됩니다.
#   Above 3 means something like 2.5 * W2 went in. The future change has mean 0, so do not stretch it.
#   첫 줄이 0 이면 아직 모르는 W5 를 넣은 것입니다. t = 2 에서는 W(5) 를 모릅니다.
#   A first line of 0 means W5 went in. At t = 2 we do not know W(5) yet.

In [ ]:
# 자가검증 / Self-check
# 위의 MSE 는 두 예측을 비교해 보여 주는 숫자입니다. 정답 판정은 함수가 하는 일을 직접 봅니다.
# The MSE above compares the two forecasts. Here we check what the function itself does.
# t = 2 에서 본 값이 w 이면 W(5) 의 조건부 평균은 w 입니다. W2 와 길이만 같은 시험용 값 w_test 를 넣어 봅니다.
# If the value seen at t = 2 is w, the conditional mean of W(5) is w. Feed in test values w_test with the same length as W2.
w_test = np.linspace(-2.0, 2.0, W2.size)
out = np.asarray(mean_forecast_of_W5(w_test))
assert out.shape == w_test.shape, \
    "예측이 입력 값마다 하나여야 합니다 / the forecast must have one value per input value"
assert not np.allclose(out, W5), \
    "아직 모르는 W5 를 예측으로 썼습니다. t = 2 에서는 W(5) 를 모릅니다 / The forecast used W5, which is not known at t = 2."
assert np.allclose(out, w_test), \
    ("t = 2 에서 본 값이 w 면 예측도 w 여야 합니다. 곱하거나 더하지 말고, 함수에 들어온 값만 쓰세요"
     " / If the value seen at t = 2 is w, the forecast must be w. Do not scale or shift it, and use only the value passed in.")
print("통과. 마팅게일이라 지금 값이 최선의 평균 예측입니다. 표준 브라운 운동이라 남는 오차의 분산은 앞으로 남은 시간 3 입니다.")
print("Passed. As a martingale, the current value is the best mean forecast. As standard Brownian motion, the error variance left is the time still to come, 3.")

In [ ]:
# 지금 값 W(2) 에 따라 미래의 변화가 달라지나? W(2) 를 다섯 구간으로 나눠 구간마다 W(5) - W(2) 의 평균을 본다.
# Does the future change depend on what we saw? Split paths by W(2) into five groups and average W(5) - W(2) in each.
edges = np.quantile(W2, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
group = np.clip(np.searchsorted(edges, W2, side="right") - 1, 0, 4)
for g in range(5):
    m = group == g
    print("W(2) 구간 / group %d  [%5.2f, %5.2f]   평균 변화 / mean of W(5) - W(2) : %6.3f"
          % (g + 1, edges[g], edges[g + 1], np.mean(W5[m] - W2[m])))
# 기대값 / Expected
#   W(2) 구간 / group 1  [-4.97, -1.20]   평균 변화 / mean of W(5) - W(2) :  0.061
#   W(2) 구간 / group 2  [-1.20, -0.35]   평균 변화 / mean of W(5) - W(2) :  0.067
#   W(2) 구간 / group 3  [-0.35,  0.36]   평균 변화 / mean of W(5) - W(2) : -0.115
#   W(2) 구간 / group 4  [ 0.36,  1.24]   평균 변화 / mean of W(5) - W(2) : -0.009
#   W(2) 구간 / group 5  [ 1.24,  4.81]   평균 변화 / mean of W(5) - W(2) : -0.049
#   다섯 구간 모두 0 근처입니다. 무엇을 보았든 앞으로의 변화는 평균 0. 이것이 마팅게일입니다.
#   All five groups sit near 0. Whatever you saw, the future change has mean 0. That is the martingale property.

plt.figure(figsize=(5.5, 4))
plt.scatter(W2, W5, s=3, alpha=0.3)
plt.plot([-5, 5], [-5, 5], "k--", lw=1, label="forecast = W(2)")
plt.xlabel("W(2), what we saw")
plt.ylabel("W(5), what came")
plt.title("the cloud sits on the diagonal")
plt.legend()
plt.show()

**What to notice.** Because $W$ is a martingale, the best mean forecast (in mean squared error) is flat at the current value. It is not certain: because $W$ is standard Brownian motion, the error that remains has variance $5 - 2 = 3$,
the time still to come. That is the fan on "The Forecast Stays the Same, but Uncertainty Grows".

> 요약: 마팅게일이면 최선의 평균 예측은 지금 값입니다. 남는 불확실성의 크기는 마팅게일이 정하지 않습니다. 표준 브라운 운동에서는 그 조건부 분산이 남은 시간과 같습니다. 평평한 예측과 넓어지는 띠는 함께 갑니다.

**Try it.** 덱 「Can a Martingale Rise 90% of the Time?」 의 예입니다. 매일 과거와 독립으로 0.9 확률로 +1, 0.1 확률로 -9 만큼 움직이면, 과거를 알아도 다음 변화의 기대값이 0 이라 그 누적합이 마팅게일입니다.
아래 한 줄을 실행하고 평균 변화와 오른 날의 비율을 비교해 보세요.
*The example on "Can a Martingale Rise 90% of the Time?": each day, independently of the past, the change is +1 with probability 0.9 and -9 with probability 0.1. Whatever the past, the expected change is 0, so the running total is a martingale.
Run the line below and compare the mean change with the share of up days.*

In [ ]:
chg = np.random.default_rng(7).choice([1.0, -9.0], p=[0.9, 0.1], size=100_000)   # 하루치 변화 10만 개 / 100,000 daily changes
print("평균 변화 / mean change :", round(chg.mean(), 3), "   오른 날 비율 / share of up days :", round(np.mean(chg > 0), 3))
# 기대값 / Expected
#   평균 변화 / mean change : -0.004    오른 날 비율 / share of up days : 0.9
#   평균은 0 근처인데 오른 날은 90% 입니다. 마팅게일은 평균만 정하고 오를 확률은 정하지 않습니다.
#   The mean is near 0 yet 90% of days go up. A martingale fixes the mean, not the chance of going up.

---
## 5. 같은 충격으로 만든 두 계열 / Two series from the same shocks

덱 「Do the Two Series Rise for the Same Reason?」 의 두 계열을 만듭니다. 충격 $\varepsilon_t$ 는 둘이 같습니다.
*We build the two series from "Do the Two Series Rise for the Same Reason?". Both use the same shocks $\varepsilon_t$.*

$$A_t = 0.5\,t + \varepsilon_t \qquad\qquad B_t = B_{t-1} + 0.5 + \varepsilon_t,\quad B_0 = 0$$

$B$ 를 풀어 쓰면 $B_t = (0.5 + \varepsilon_1) + (0.5 + \varepsilon_2) + \dots + (0.5 + \varepsilon_t)$ 입니다. 덱 「Series B Keeps Every Past Shock」.
*Unrolled, $B_t$ is the running total of $0.5 + \varepsilon$. See "Series B Keeps Every Past Shock".*

In [ ]:
# ---------------------------------------------------------------- TODO 3 ----
# 다음 이름을 씁니다 / Names used below
#   T    : 계열 길이. 시각 1 부터 T 까지 / series length, times 1 to T
#   eps  : 충격 ε_1 … ε_T. 두 계열이 같은 충격을 쓴다 / the shocks, shared by both series
#   t    : 시각 1, 2, …, T / the times
#   A    : 추세정상 계열. 정해진 직선 0.5 t 에 그때의 충격을 더한다
#          trend-stationary series. The fixed line 0.5 t plus the shock of that moment
#   B    : 차분정상 계열. 지난 값에 0.5 와 새 충격을 더해 간다   <- 여기가 빈칸
#          difference-stationary series. Each step adds 0.5 and a new shock to the last value   <- this is the blank
#
# 참고 / Reference   np.cumsum(x)
#   누적합. k 번째 값은 x 의 처음부터 k 번째까지를 더한 것이다.
#   Running total. The k-th value is the sum of the first k entries of x.
#   예 / e.g.   np.cumsum([1, 2, 3])  ->  [1, 3, 6]

def build_B(eps):
    """Return B_1 … B_T for B_t = B_{t-1} + 0.5 + eps_t with B_0 = 0."""
    # TODO 3. 풀어 쓰면 B_t = (0.5 + ε_1) + … + (0.5 + ε_t) 입니다. 이 누적합을 한 줄로 만들어 B 에 넣으세요.
    # TODO 3. Unrolled, B_t = (0.5 + ε_1) + … + (0.5 + ε_t). Build that running total in one line and put it into B.
    B = None
    if B is None:
        raise NotImplementedError(BLANK_MSG)

    return B
# --------------------------------------------------------------------------


rng = np.random.default_rng(101)                            # 덱 그림과 같은 시드 / same seed as the deck's figures
T = 100
eps = rng.standard_normal(T)                                # 충격. 평균 0, 분산 1, 서로 독립 / shocks: mean 0, variance 1, independent
t = np.arange(1, T + 1)                                     # 시각 1 … T / times 1 … T
A = 0.5 * t + eps                                           # 추세정상 / trend-stationary
B = build_B(eps)                                            # 차분정상 / difference-stationary

print("B 의 처음 세 값 / first three values of B :", np.round(B[:3], 3))
print("0.5 + eps 의 누적 / running total by hand  :", np.round([0.5 + eps[0], 1.0 + eps[0] + eps[1],
                                                          1.5 + eps[0] + eps[1] + eps[2]], 3))
# 기대값 / Expected
#   B 의 처음 세 값 / first three values of B : [-0.29  -1.825 -0.721]
#   0.5 + eps 의 누적 / running total by hand  : [-0.29  -1.825 -0.721]
#   두 줄이 같으면 맞습니다. 첫 값은 0.5 + ε_1 입니다(B_0 = 0 에서 한 걸음).
#   The two lines agree when it is right. The first value is 0.5 + ε_1, one step from B_0 = 0.
#   첫 줄이 [0.5 1. 1.5] 처럼 충격 없이 곧게 오르면 eps 를 빼먹은 것입니다.
#   A first line that climbs straight, like [0.5 1. 1.5], means eps was left out.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def build_B(eps):
    """Return B_1 … B_T for B_t = B_{t-1} + 0.5 + eps_t with B_0 = 0."""
    B = np.cumsum(0.5 + eps)     # 빈칸이었던 줄. 걸음마다 0.5 + 충격을 더해 간 누적합
                                 # the blank. The running total of 0.5 + shock, one step at a time
    if B is None:
        raise NotImplementedError(BLANK_MSG)

    return B



rng = np.random.default_rng(101)                            # 덱 그림과 같은 시드 / same seed as the deck's figures
T = 100
eps = rng.standard_normal(T)                                # 충격. 평균 0, 분산 1, 서로 독립 / shocks: mean 0, variance 1, independent
t = np.arange(1, T + 1)                                     # 시각 1 … T / times 1 … T
A = 0.5 * t + eps                                           # 추세정상 / trend-stationary
B = build_B(eps)                                            # 차분정상 / difference-stationary

print("B 의 처음 세 값 / first three values of B :", np.round(B[:3], 3))
print("0.5 + eps 의 누적 / running total by hand  :", np.round([0.5 + eps[0], 1.0 + eps[0] + eps[1],
                                                          1.5 + eps[0] + eps[1] + eps[2]], 3))
# 기대값 / Expected
#   B 의 처음 세 값 / first three values of B : [-0.29  -1.825 -0.721]
#   0.5 + eps 의 누적 / running total by hand  : [-0.29  -1.825 -0.721]
#   두 줄이 같으면 맞습니다. 첫 값은 0.5 + ε_1 입니다(B_0 = 0 에서 한 걸음).
#   The two lines agree when it is right. The first value is 0.5 + ε_1, one step from B_0 = 0.
#   첫 줄이 [0.5 1. 1.5] 처럼 충격 없이 곧게 오르면 eps 를 빼먹은 것입니다.
#   A first line that climbs straight, like [0.5 1. 1.5], means eps was left out.

In [ ]:
# 자가검증 / Self-check
assert B.shape == eps.shape, "B 는 충격 eps 와 길이가 같아야 합니다 / B must have the same length as the shocks eps"
assert np.isclose(B[0], 0.5 + eps[0]), "첫 값은 0.5 + ε_1 이어야 합니다 / the first value must be 0.5 + ε_1"
assert np.allclose(np.diff(B), 0.5 + eps[1:]), \
    "한 걸음의 변화가 0.5 + 새 충격이어야 합니다 / each step must change B by 0.5 + the new shock"
print("통과. B 는 지나간 충격을 전부 담은 누적합입니다.")
print("Passed. B is a running total that keeps every past shock.")

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(t, A, label="A, fixed line + shock")
ax.plot(t, B, label="B, accumulated shocks")
ax.plot(t, 0.5 * t, "k--", lw=1, label="0.5 t")
ax.set_xlabel("time t")
ax.set_title("same shocks, same average slope")
ax.legend()
plt.show()

### 충격 하나만 바꾸면 / Change one shock

덱 「Does a One-Time Shock Disappear or Persist?」 의 실험입니다. $t = 50$ 의 충격에만 2 를 더하고, 새 계열에서 옛 계열을 뺍니다.
*The experiment on "Does a One-Time Shock Disappear or Persist?". Add 2 to the shock at $t = 50$ only and subtract the old series from the new one.*

**먼저 예측하세요.** $t = 50, 51, 52, 53$ 에서 A 와 B 의 차이는 각각 얼마일까요?
*Predict first. What are the differences for A and for B at $t = 50, 51, 52, 53$?*

In [ ]:
# 5절의 충격을 다시 만든다. 6절이 같은 이름을 덮어쓰므로, 이 셀만 따로 돌려도 같은 결과가 나오게.
# Rebuild the section 5 shocks. Section 6 reuses these names, so this keeps the cell self-contained.
eps = np.random.default_rng(101).standard_normal(100)
t = np.arange(1, 101)
A, B = 0.5 * t + eps, build_B(eps)

eps2 = eps.copy()
eps2[49] += 2.0                                             # 배열 번호는 0 부터라 t = 50 은 49 번 칸 / index 49 is t = 50
A2 = 0.5 * t + eps2
B2 = build_B(eps2)                                          # 5절에서 만든 함수를 그대로 쓴다 / reuse the function from above
print("t      :", t[49:53])
print("A 차이 / A difference :", np.round((A2 - A)[49:53], 3))
print("B 차이 / B difference :", np.round((B2 - B)[49:53], 3))
print("B 차이, 마지막 / B difference at t = 100 :", round((B2 - B)[-1], 3))
# 기대값 / Expected
#   t      : [50 51 52 53]
#   A 차이 / A difference : [2. 0. 0. 0.]
#   B 차이 / B difference : [2. 2. 2. 2.]
#   B 차이, 마지막 / B difference at t = 100 : 2.0
#   A 는 50 에서만 2 이고 그 뒤로 0 입니다. B 는 50 부터 끝까지 계속 2 입니다.
#   A differs only at t = 50. B differs by 2 from t = 50 to the very end.

**What to notice.** In A, the effect of the shock disappears after one step. In B, the effect remains in every future value, because $B$ is a running total of shocks.

> 요약: 같은 충격의 효과가 A 에서는 한 걸음 뒤 사라지고, B 에서는 이후 모든 값에 남습니다. 추세를 어떻게 빼야 하는지가 여기서 갈립니다.

---
## 6. 두 변환, 네 조합 / Two transformations, four combinations

추세를 빼는 방법은 둘입니다. 정해진 직선을 빼는 **추세 제거**, 한 칸 앞과의 차이를 보는 **차분** $\nabla x_t = x_t - x_{t-1}$.
덱 「Remove Each Trend the Way It Was Built」 · 「Series A Can Be Differenced Too. But Should It Be?」 · 「Differencing A Creates a New Correlation」 가 이 절입니다.
*Two ways to remove a trend: **detrending** subtracts the fixed line, **differencing** takes $\nabla x_t = x_t - x_{t-1}$.
This section is the deck slides "Remove Each Trend the Way It Was Built", "Series A Can Be Differenced Too. But Should It Be?" and "Differencing A Creates a New Correlation".*

자기상관은 2주차의 표본 ACF 로 잽니다. 여기서는 시차 1 만 봅니다. 계열이 길수록 숫자가 안정되므로 길이를 1000 으로 늘립니다.
*We measure dependence with Week 2's sample ACF, at lag 1 only. A longer series gives steadier numbers, so the length goes up to 1000.*

**먼저 예측하세요.** 네 조합 가운데 정상으로 보일 것은? 시차 1 자기상관이 $-0.5$ 근처인 것과 1 근처인 것은?
*Predict first. Which of the four will look stationary? Which one will have a lag-1 autocorrelation near $-0.5$, and which one near 1?*

In [ ]:
rng = np.random.default_rng(1)                              # 덱 Lab 2 와 같은 시드 / same seed as the deck's Lab 2
T = 1000
eps = rng.standard_normal(T)
t = np.arange(1, T + 1)
A = 0.5 * t + eps                    # trend-stationary
B = build_B(eps)                     # difference-stationary, section 5
series = {'detrend A': A - 0.5 * t, 'diff A': np.diff(A),
          'detrend B': B - 0.5 * t, 'diff B': np.diff(B)}
for name, x in series.items():
    x = x - x.mean()
    print(name, round((x[1:] * x[:-1]).sum() / (x * x).sum(), 2))
# 기대값 / Expected
#   detrend A -0.02
#   diff A -0.5
#   detrend B 1.0
#   diff B -0.02
#   마지막 줄의 식은 2주차 표본 ACF 의 시차 1 값입니다. 평균을 빼고, 이웃끼리 곱한 합을 제곱합으로 나눕니다.
#   The last line is Week 2's sample ACF at lag 1: subtract the mean, then the sum of neighbour products over the sum of squares.
#   추세 제거한 A 와 차분한 B 는 0 근처. 차분한 A 는 -0.5 근처. 추세 제거한 B 는 1 근처입니다.
#   Detrended A and differenced B sit near 0. Differenced A sits near -0.5. Detrended B sits near 1.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 5), sharex=True)
for a_, (name, x) in zip(ax.ravel(), series.items()):
    a_.plot(x[:300], lw=0.8)
    a_.set_title(name)
for a_ in ax[1]:
    a_.set_xlabel("time, first 300 steps")
plt.tight_layout()
plt.show()

**What to notice.** Detrending suits A and differencing suits B: each leaves the shock $\varepsilon_t$ (plus 0.5 for B).
Detrending B leaves a random walk, which wanders and has lag-1 autocorrelation near 1. Differencing A is stationary but creates
$\varepsilon_t - \varepsilon_{t-1}$, whose lag-1 autocorrelation is $-\sigma^2 / 2\sigma^2 = -1/2$, a dependence that was not there.
This is called over-differencing. A stationary result does not mean differencing was the right choice.

Three of the four are stationary: detrended A, differenced A and differenced B. A lag-1 value near 1 is a clue that a series may not be
stationary, not a test. The four numbers come from seed 1; another seed gives slightly different values.
How many times to difference is the $d$ of ARIMA in Week 7.

> 요약: A 는 추세 제거, B 는 차분이 맞습니다. B 를 추세 제거하면 랜덤워크가 남고(ρ(1) ≈ 1), A 를 차분하면 없던 음의 상관(ρ(1) ≈ -0.5)이 생깁니다. 이것을 과차분(over-differencing)이라 하고, 결과가 정상이어도 차분이 맞는 선택이었다는 뜻은 아닙니다. 정상인 것은 추세 제거한 A · 차분한 A · 차분한 B 셋입니다. 1 근처는 비정상을 의심하게 하는 단서이지 판정이 아니고, 숫자는 시드 1 의 값입니다. 몇 번 차분해야 하는지가 7주차 ARIMA 의 d 입니다.

**Try it. 덱 그림 다시 그리기.** 덱 「Remove Each Trend the Way It Was Built」 의 두 그림은 5절의 계열(시드 101, 길이 100)로 그린 것입니다. 위의 네 그림은 Lab 2 의 계열(시드 1, 길이 1000)이라 모양이 다릅니다. 아래 셀이 덱과 같은 두 그림을 그립니다.
*The two panels on "Remove Each Trend the Way It Was Built" use the section 5 series (seed 101, length 100). The four panels above use the Lab 2 series (seed 1, length 1000), so they look different. The cell below draws the deck's two panels.*

In [ ]:
# 덱 「Remove Each Trend the Way It Was Built」 의 두 그림. 5절과 같은 충격으로 두 계열을 다시 만든다.
# The two panels on "Remove Each Trend the Way It Was Built", rebuilt from the section 5 shocks.
eps = np.random.default_rng(101).standard_normal(100)       # 5절과 같은 충격 / the section 5 shocks
t = np.arange(1, 101)                                       # 시각 1 … 100 / times 1 … 100
A, B = 0.5 * t + eps, build_B(eps)                          # 5절의 A 와 B. build_B 는 5절의 빈칸 / the section 5 series; build_B is the section 5 blank
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(t, A - 0.5 * t, color="darkgreen")               # 직선을 빼면 충격만 남는다 / subtract the line and the shock is left
ax[0].axhline(0, color="0.5", ls="--", lw=1)
ax[0].set_title("detrend A:  A_t - 0.5 t")
ax[0].set_xlabel("time t")
ax[1].plot(t[1:], np.diff(B), color="firebrick")            # 한 칸 차이는 0.5 + 새 충격 / each step is 0.5 plus one new shock
ax[1].axhline(0.5, color="0.5", ls="--", lw=1)
ax[1].set_title("difference B:  B_t - B_(t-1)")
ax[1].set_xlabel("time t")
plt.tight_layout()
plt.show()
# 기대값 / Expected
#   덱의 두 그림과 같은 모양입니다. 왼쪽은 0 둘레에서, 오른쪽은 0.5 둘레에서 흔들립니다.
#   The same shapes as the deck's two panels. The left one wobbles around 0, the right one around 0.5.

---
## 7. Optional / For the curious

덱 「Continuous Does Not Mean Smooth」 와 다음 주 예고입니다. 건너뛰어도 됩니다.
*The deck slide "Continuous Does Not Mean Smooth", and a look at next week. Safe to skip.*

In [ ]:
#@title ▶ Optional A. 확대해도 기울기가 정해지지 않는다 / Zoom in and the slope does not settle { display-mode: "form" }
# 길이 h 인 걸음의 기울기 (W(t+h) - W(t)) / h 의 분산은 h / h^2 = 1 / h. h 를 줄이면 오히려 커진다.
# The slope over a step of length h has variance h / h^2 = 1 / h. Shrinking h makes it larger, not smaller.
g = np.random.default_rng(3)
for h in (0.1, 0.01, 0.001):
    slope = np.sqrt(h) * g.standard_normal(100_000) / h     # 걸음 하나의 변화를 h 로 나눈다 / one step's change over h
    print("h = %-6g  기울기의 분산 / variance of the slope : %9.1f    1/h = %g" % (h, slope.var(), 1 / h))

In [ ]:
#@title ▶ Optional B. 다음 주. 변환 뒤에 남는 의존 / Next week, the dependence that remains { display-mode: "form" }
# u_t = 0.5 u_{t-1} + ε_t. 오늘은 어제의 절반을 이어받는다. 자기상관은 시차마다 절반씩 준다.
# u_t = 0.5 u_{t-1} + ε_t. Today keeps half of yesterday, so the autocorrelation halves at each lag.
g = np.random.default_rng(303)
u = np.zeros(20_000)
z = g.standard_normal(20_000)
for k in range(1, len(u)):
    u[k] = 0.5 * u[k - 1] + z[k]
x = u - u.mean()
for lag in (1, 2, 3):
    r = (x[lag:] * x[:-lag]).sum() / (x * x).sum()
    print("시차 / lag %d   표본 / sample %.3f   이론 / theory 0.5^%d = %.3f" % (lag, r, lag, 0.5 ** lag))

---
## 8. 직접 해 볼 자리 / Scratch space

2절의 `np.sqrt(dt)` 를 바꿔 보거나, 5절에서 $\varepsilon_{50}$ 에 더하는 2 를 바꿔 보세요. 바꾼 뒤에는 원래대로 되돌리고 그 셀을 다시 실행하세요.
*Change `np.sqrt(dt)` in section 2, or the +2 added to $\varepsilon_{50}$ in section 5. Change it back and re-run that cell afterwards.*

In [ ]:
# 8절 연습 공간 / Scratch space for section 8.

---
### 이번 주에 가져갈 것 / What to take away

1. 분산이 일정한 독립 충격이 쌓이면 분산은 $t$ 로, 퍼짐은 $\sqrt{t}$ 로 큽니다. 걸음이 $\sqrt{h}$ 에 비례해야 그렇게 되고, 정확히 $\sqrt{h}$ 면 분산이 $t$ 그대로입니다.
   *When independent shocks with a constant variance accumulate, the variance grows like $t$ and the spread like $\sqrt{t}$. That needs steps proportional to $\sqrt{h}$; exactly $\sqrt{h}$ gives variance $t$.*
2. 두 시각의 공분산은 공유한 역사의 길이 $\min(s, t)$ 입니다.
   *The covariance of two times is the length of their shared history, $\min(s, t)$.*
3. 마팅게일이면 조건부 평균 예측(제곱오차 기준 최선)은 지금 값입니다. 오를 확률도, 남는 불확실성의 크기도 마팅게일이 정하지 않습니다. 표준 브라운 운동에서는 조건부 분산이 남은 시간과 같습니다.
   *For a martingale, the conditional mean forecast is the current value (best in squared error). It fixes neither the chance of going up nor the size of the remaining uncertainty. For standard Brownian motion, the conditional variance is the remaining time.*
4. A 처럼 고정 추세에 정상 편차가 붙은 계열은 추세 제거, B 처럼 충격이 쌓인 수준은 차분. 반대로 하면 랜덤워크가 남거나 없던 음의 상관이 생깁니다. 충격이 사라져도 뺄 추세가 없는 계열(다음 주의 AR(1))도 있으니, 「충격이 사라지면 추세 제거」로 외우지 마세요.
   *Detrend a fixed trend with stationary deviations, like A; difference an accumulated level, like B. The wrong one leaves a random walk or creates a negative correlation. Some series have fading shocks and no trend to remove (next week's AR(1)), so do not read this as "fading shocks mean detrend".*
5. 실제 계열을 분석할 때 오늘 배운 것은 세 판단으로 들어갑니다(덱 「Where Today's Ideas Enter a Time-Series Analysis」). 추세를 직선으로 뺄지 차분할지 정하기(6절. 틀리면 랜덤워크가 남거나 없던 음의 상관이 생김). 「내일 = 오늘」을 비교 기준 예측으로 두고 모형이 그보다 나은지 보기(4절. 계열이 마팅게일이면 과거만 쓰는 어떤 예측도 평균제곱오차로 이 기준보다 나을 수 없고, 기준이 없으면 모형이 무엇을 더했는지 알 수 없음). 랜덤워크처럼 충격이 쌓이는 계열이면 먼 미래일수록 예측 구간을 $\sqrt{h}$ 로 넓히기(2·4절. 정상 계열은 폭이 일정 수준에 머묾. 넓히지 않으면 먼 구간이 너무 좁음).
   *On a real series, today's ideas enter three decisions (deck "Where Today's Ideas Enter a Time-Series Analysis"). Decide whether to subtract a line or difference (section 6; the wrong choice leaves a random walk or creates a negative correlation). Keep tomorrow = today as the reference forecast and check whether a model does better (section 4; for a martingale no forecast built from the past has a smaller mean squared error, and without the reference you cannot tell what a model adds). For a random-walk-like series, where shocks accumulate, widen the forecast interval like $\sqrt{h}$ the further ahead you go (sections 2 and 4; for a stationary series the width levels off, and not widening makes far intervals too narrow).*

마지막 셀은 실행 환경을 적는 기록용 셀입니다. 이 노트북은 제출하지 않습니다.
*The last cell records the environment. There is nothing to submit for this notebook.*

In [ ]:
import sys, platform, datetime
print("date    :", datetime.datetime.now().isoformat(timespec="seconds"))
print("python  :", sys.version.split()[0], "on", platform.platform())
print("numpy   :", np.__version__)
import matplotlib
print("mpl     :", matplotlib.__version__)
print("seeds   : 5, 8, 9, 11, n (1) · 20260929 (2-4) · 7 (4) · 101 (5, 6) · 1 (6) · 3, 303 (7)")